# Module 3: Paired RNA-ATAC multiome and MOFA

**Biological question.** In Donor_2 SNARE-seq2 multiome (`HBM828.GPVG.252`), do MOFA factors capture shared RNA-ATAC structure, and if not, did the method fail or is the ATAC gene-activity view empty of cell type?

**Learning objectives.** Analyze a fitted multiome factor model; distinguish joint latent structure from modality-private factors; tell a broken model from an absent ATAC signal (LO4).

**Bloom.** Analyze

**Prerequisites.** Module 1 (QC objects) and Module 2 (integrated HuBMAP object / pseudobulk exports). Conda env `CFDE_lung_env` (or the pinned pip venv). Default path uses the fitted MOFA HDF5 plus the committed label cache; full MuData is not a learner download.

**Time.** Instructional: 30 min. Compute is short on the default path. Measured peak RSS 7.1 GB; 16 GB recommended, 8 GB is the minimum.

**Role in the course.** This module reads two modalities measured in the same nucleus. Modules 1, 2 and 4 compare separate transcriptomic datasets, so this is where the question changes from whether datasets agree to whether two measurements of one nucleus agree. The Donor_2 10x snRNA teaching block `HBM473.NKMR.872` shares a block and a section with this multiome (different nuclei; no per-nucleus join).

**Data (default path).** `multiome_mofa.hdf5` plus `outputs/tables/module3_label_cache_HBM828.GPVG.252.tsv`. Full `secondary_analysis.h5mu` is an optional authoring extension (`module3.params.load_h5mu`) and is not downloaded by learners.

It does not re-fit MOFA inside the lesson.


## How this module fits the course

Modules 1 and 2 measure one modality, RNA, across four donors. Module 4 compares four separate
resources. This module is the only place the course reads two modalities measured in the **same
nucleus**, so it is where the question changes from "do these datasets agree" to "do these two
measurements of one nucleus agree".

The notebook works through six questions in order. Each one is answerable from the tables it prints,
and each section ends with the finding it established.

| Section | Question | What it establishes |
|---|---|---|
| 1 | What is this object? | Provenance, nuclei, the matched 10x block, access tiers |
| 2 | Does MOFA find a shared axis? | Variance explained per view, factor dominance |
| 3 | Did MOFA work at all? | Whether factors track cell identity (the positive control) |
| 4 | What is in the ATAC view? | Cluster concordance, and what the ATAC clustering encodes |
| 5 | Does WNN do better? | A second integration method on the same nuclei |
| 6 | Gene bridge and focus genes | Feature-level agreement, global and within cell type |

Read them as one argument rather than six results. A negative result about an integration method is
only interpretable once you have shown the method works where signal exists, which is why section 3
comes before section 4.

**No large download is required.** Sections 3 to 5 need per-nucleus cell-type labels and the WNN
clustering, which are read from a small committed cache rather than the multi-gigabyte MuData object.
Section 1 records where that cache came from.


## 0. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

from IPython.display import display

MODULE_ROOT = Path.cwd().resolve()
if MODULE_ROOT.name == "notebooks":
    MODULE_ROOT = MODULE_ROOT.parent
if not (MODULE_ROOT / "scripts").is_dir():
    MODULE_ROOT = next(
        (p for p in MODULE_ROOT.parents if (p / "scripts").is_dir()), MODULE_ROOT
    )
os.chdir(MODULE_ROOT)
sys.path.insert(0, str(MODULE_ROOT))

import pandas as pd

from scripts.common.paths import ensure_output_dirs, load_config, resolve
from scripts.common.runtime import finalize_timing, peak_rss_mb
from scripts.nb05_multimodal.bridge import (
    azimuth_composition,
    barcode_overlap_from_mofa,
    barcode_overlap_table,
    cluster_concordance,
    combine_focus_correlations,
    factor_label_association,
    focus_gene_availability,
    focus_gene_bridge,
    focus_gene_correlations,
    focus_gene_correlations_by_cell_type,
    focus_from_paired_matrix,
    modality_contrast_table,
    mofa_factor_dominance,
    qc_association,
    rna_atac_gene_bridge,
)
from scripts.nb05_multimodal.export import save_module3_outputs
from scripts.nb05_multimodal.inventory import inventory_module3_assets
from scripts.nb05_multimodal.load import (
    adata_from_mofa_factors,
    assemble_nucleus_frame,
    focus_gene_matrix,
    focus_gene_matrix_from_mofa,
    load_joint_embedding,
    load_label_cache,
    load_mofa,
    mofa_nondominant_r2_summary,
    mofa_sample_metadata,
    multiome_paths,
    recompute_mofa_r2_total,
)
from scripts.nb05_multimodal.plots import (
    plot_cluster_concordance,
    plot_factor_label_heatmap,
    plot_focus_correlations,
    plot_focus_gene_bars,
    plot_mofa_umap,
    plot_mofa_variance,
    plot_qc_association,
    plot_rna_atac_scatter,
    plot_umap_panel,
    plot_wnn_umap,
)

cfg = load_config()
ensure_output_dirs(cfg)
fig_dir = resolve(cfg, "outputs_figures")
paths = multiome_paths(cfg)
donor = cfg["module3"]["donor_label"]
load_h5mu = bool(cfg["module3"].get("params", {}).get("load_h5mu", False))
min_n = int((cfg["module3"].get("thresholds") or {}).get("min_nuclei_correlation", 50))
noise_delta = float((cfg["module3"].get("thresholds") or {}).get("noise_delta_r", 0.01))
max_fac = int((cfg["module3"].get("analysis") or {}).get("mofa_variance_max_factor", 18))
_m3_timing = {
    "_t0": time.perf_counter(),
    "_compute_seconds": None,
    "peak_rss_mb": None,
    "peak_rss_mb_start": peak_rss_mb(),
    "peak_rss_note": (
        "process peak RSS so far (ru_maxrss); monotone across a loop, not per-step"
    ),
}
print("donor:", donor, "dataset:", cfg["module3"]["dataset_id"])
print("matched snRNA:", (cfg["module3"].get("matched_snrna") or {}).get("primary_id"))
print("load_h5mu:", load_h5mu)
mofa = load_mofa(cfg)
nucleus = assemble_nucleus_frame(cfg)
print("nuclei:", nucleus.shape[0], "cache labels:", nucleus["azimuth_label"].nunique())
display(inventory_module3_assets(cfg)[["file_name", "exists", "size_mb", "core_or_optional", "notes"]])


## 1. Object composition and barcode pairing

Donor_2 SNARE-seq2 product `HBM828.GPVG.252` is paired RNA + ATAC in the same nuclei. It shares a block and a section with the Module 1-2 10x snRNA teaching block `HBM473.NKMR.872`. Those are different nuclei, so there is no barcode join across assays.

Access tiers: learners download the MOFA HDF5; the Azimuth / WNN cache ships in the package; the ~14.7 GiB MuData file is not a learner download.


In [ ]:
if load_h5mu and paths["secondary_h5mu"].exists():
    overlap = barcode_overlap_table(cfg)
    adata = load_joint_embedding(cfg)
    focus_long = focus_gene_matrix(cfg, mofa)
else:
    overlap = barcode_overlap_from_mofa(mofa)
    adata = adata_from_mofa_factors(mofa)
    focus_long = focus_gene_matrix_from_mofa(cfg, mofa)

for col in ("azimuth_label", "leiden_wnn", "rna_leiden", "atac_leiden"):
    if col in nucleus.columns:
        adata.obs[col] = nucleus.reindex(adata.obs_names)[col].to_numpy()
if not focus_long.empty and "azimuth_label" not in focus_long.columns:
    focus_long = focus_long.merge(
        nucleus.reset_index()[["barcode", "azimuth_label"]],
        on="barcode",
        how="left",
    )

comp = azimuth_composition(nucleus)
display(overlap)
display(comp.head(12))
display(modality_contrast_table(cfg))
print("RNA Leiden k=", nucleus["rna_leiden"].nunique(),
      "ATAC Leiden k=", nucleus["atac_leiden"].nunique(),
      "ArchR k=", nucleus["atac_clusters"].nunique(),
      "WNN k=", nucleus["leiden_wnn"].nunique())


## 2. Variance explained across modalities

Shared latent biology would show factors with substantial R2 in **both** views. Percent-scale stored totals mean 1.3 is 1.3 percent, not a fraction above 1.


In [ ]:
r2_recompute = recompute_mofa_r2_total(cfg, view="rna")
nondom = mofa_nondominant_r2_summary(mofa["variance_per_factor"], r2_recompute=r2_recompute)
factor_dom = mofa_factor_dominance(mofa["variance_per_factor"], n_factors=max_fac)
print(r2_recompute.get("label"))
print(
    f"Max non-dominant R2 = {nondom.get('max_nondominant_r2'):.5f} "
    f"({nondom.get('max_nondominant_percent'):.4f} percent)"
)
display(mofa["variance_total"])
display(factor_dom)
plot_mofa_variance(
    mofa["variance_total"],
    mofa["variance_per_factor"],
    fig_dir / "module3_mofa_variance.png",
    n_factors=max_fac,
    r2_recompute=r2_recompute,
)
plot_mofa_umap(adata, fig_dir / "module3_mofa_latent.png", donor_label=donor)


**Finding.** No. Every active factor is modality-private, and trailing factors are numerically dead. Check `module3_mofa_factor_dominance.tsv`.


## 3. Factor association with cell identity

A private-factor model can still have found biology in one view. Eta squared of each factor score against cell identity is the check.


In [ ]:
assoc = factor_label_association(
    mofa["factors"],
    nucleus,
    label_cols=["azimuth_label", "leiden_wnn", "rna_leiden", "atac_leiden"],
)
display(
    assoc.sort_values("eta_squared", ascending=False)
    .groupby("labelling", as_index=False)
    .first()
)
display(assoc[assoc["factor"].isin(["Factor6", "Factor3", "Factor4"])])
plot_factor_label_heatmap(
    assoc, fig_dir / "module3_factor_label_heatmap.png", n_factors=max_fac
)


**Finding.** Yes, in RNA. RNA-dominant factors track Azimuth / WNN / RNA Leiden and sit near the floor against ATAC Leiden. See `module3_factor_label_association.tsv`.


## 4. Content of the ATAC gene-activity view

If ATAC Leiden is chance-level against cell type and strongly associated with depth, the gene-activity matrix is not a basis for integration.


In [ ]:
conc = cluster_concordance(nucleus)
qc_df = qc_association(
    nucleus,
    label_cols=["atac_leiden", "rna_leiden"],
    qc_cols=["nFrags", "ReadsInTSS", "DoubletEnrichment", "PromoterRatio"],
)
display(conc)
display(qc_df)
plot_cluster_concordance(nucleus, fig_dir / "module3_cluster_concordance.png")
plot_qc_association(qc_df, fig_dir / "module3_qc_association.png")


**Finding.** ATAC Leiden is near chance against Azimuth and tracks log10 fragment depth. RNA Leiden does not. The emptiness is technical. Tables: `module3_cluster_concordance.tsv`, `module3_qc_association.tsv`.


## 5. Weighted nearest neighbor concordance

WNN is the portal's second integrator. A joint embedding can look successful while still leaning on RNA.


In [ ]:
display(conc[conc["col_a"].eq("leiden_wnn") | conc["col_b"].eq("leiden_wnn")])
plot_wnn_umap(nucleus, fig_dir / "module3_wnn_umap.png", donor_label=donor)


**Finding.** WNN follows RNA Leiden more than ATAC Leiden, and it agrees with Azimuth less than RNA Leiden does. The joint embedding is not a rescue of the ATAC view.


## 6. Gene bridge and focus genes

ATAC gene activity is not RNA expression. The cache now supplies Azimuth labels, so within-cell-type correlations are computed on the default path and flagged when n is below 50.


In [ ]:
bridge = rna_atac_gene_bridge(mofa)
focus = focus_from_paired_matrix(focus_long)
if focus.empty:
    focus = focus_gene_bridge(bridge, cfg)
corr_global = focus_gene_correlations(focus_long, min_nuclei=min_n)
corr_cell = focus_gene_correlations_by_cell_type(focus_long, cfg)
corr = combine_focus_correlations(corr_global, corr_cell)
availability = focus_gene_availability(cfg, focus_long, bridge)
print(
    "shared genes", bridge.shape[0],
    "r", bridge.attrs.get("pearson_log1p"),
    "rho", bridge.attrs.get("spearman_log1p"),
)
display(corr)
display(availability)
plot_rna_atac_scatter(
    bridge, fig_dir / "module3_rna_atac_scatter.png", focus_df=focus, donor_label=donor
)
plot_focus_correlations(
    corr, fig_dir / "module3_focus_gene_correlations.png", noise_delta=noise_delta
)
plot_focus_gene_bars(focus, fig_dir / "module3_focus_gene_bars.png")

finalize_timing(_m3_timing)
contrast = modality_contrast_table(cfg)
inventory = inventory_module3_assets(cfg)
out = save_module3_outputs(
    cfg,
    inventory_df=inventory,
    contrast_df=contrast,
    overlap_df=overlap,
    bridge_df=bridge,
    focus_df=focus,
    corr_df=corr,
    variance_total=mofa["variance_total"],
    variance_per_factor=mofa["variance_per_factor"],
    figure_paths={
        "mofa_variance": fig_dir / "module3_mofa_variance.png",
        "mofa_latent": fig_dir / "module3_mofa_latent.png",
        "qc_association": fig_dir / "module3_qc_association.png",
        "factor_label_heatmap": fig_dir / "module3_factor_label_heatmap.png",
        "wnn_umap": fig_dir / "module3_wnn_umap.png",
        "cluster_concordance": fig_dir / "module3_cluster_concordance.png",
        "rna_atac_scatter": fig_dir / "module3_rna_atac_scatter.png",
        "focus_correlations": fig_dir / "module3_focus_gene_correlations.png",
    },
    extras={
        "multiome_dir": str(paths["dir"]),
        "mofa_hdf5": str(paths["mofa_hdf5"]),
        "n_nuclei": int(adata.n_obs),
        "n_mofa_factors": int(mofa["factors"].shape[1]),
        "portal_url": cfg["module3"].get("portal_url"),
        "load_h5mu": load_h5mu,
        **{k: v for k, v in _m3_timing.items() if not str(k).startswith("_t")},
    },
    availability_df=availability,
    factor_dom_df=factor_dom,
    cluster_concordance_df=conc,
    qc_association_df=qc_df,
    factor_label_association_df=assoc,
    azimuth_composition_df=comp,
    r2_recompute=r2_recompute,
    nondom_summary=nondom,
)
print("Wrote", out["interpretation"])


## Questions this result raises

1. What evidence shows this is true multiome rather than unpaired same-donor assays, and what does the matched 10x block license that Donor_3 could not?
2. Why is this MOFA space not a joint latent space? Quote the max non-dominant R2 from the table.
3. What result shows that MOFA still found biology? Quote an eta squared for an RNA factor against Azimuth.
4. Why is the ATAC gene-activity view not a legitimate basis for integration on this object?
5. WNN looks like a joint embedding. What comparison shows that it is leaning on RNA?
6. When a within-cell-type focus-gene row is flagged for low n, what claim is still licensed?
7. Factor1 carries the largest single share of ATAC variance yet tracks no cell-type label, no cluster assignment and no recorded QC metric, while sitting at eta squared 0.488 against the WNN clustering. WNN takes the ATAC representation as an input. What does that shared input explain, and what does it leave open?
